# Concrete Crack Detection and Localization Using Deep Learning

## Project Proposal - Data Science Workshop

**Submitted by:** _[name 1, ID]_, _[name 2, ID]_, _[name 3, ID]_

**Supervisor:** _[supervisor name]_

---

### What this notebook is

This is the Stage 1 project proposal. It is the skeleton of the final report: every
chapter that will appear in the final project appears here, already populated with
real results rather than intentions. Where a step is expensive we run it on a clearly
labelled subset and say so; nothing in this notebook is a placeholder.

The notebook runs top to bottom with Kernel > Restart and Run All. All randomness is
seeded, and all paths are relative to the repository root.

---
# 1. Problem definition and project goal

### The problem

Concrete structures crack. Cracks are the earliest visible symptom of structural
distress, and their **location, length and width** determine whether a crack is
cosmetic or the leading edge of a failure. Today this assessment is made by an
engineer physically inspecting the surface, which is slow, expensive, requires
access to the structure, and is subjective enough that two inspectors can disagree.

### The task

We frame this as a **semantic segmentation** problem - a single prediction task:

```
RGB image patch  ->  segmentation model  ->  binary mask
                                             0 = background
                                             1 = crack
```

The model receives a patch of a concrete photograph and outputs a mask assigning a
label to **every pixel**. A patch containing no crack has an empty mask as its
target; the model is expected to return an empty mask for it.

We deliberately do **not** build a separate crack / no-crack classifier. Pixel-level
localization strictly contains the presence question, and splitting the work into two
models would mean two things to train, tune and defend for no added insight. Crack
statistics computed from the masks in this notebook are **descriptive only** - they
characterise the target distribution and are never fed to the model as inputs.

### Research questions

1. Can a model localize cracks at the pixel level accurately enough to be useful,
   given only 458 annotated photographs?
2. How much does a deep segmentation model actually gain over classical image
   processing? Cracks are dark, thin, curvilinear structures - exactly what
   morphological filters were designed for, so this is a real question, not a
   formality.
3. Where does the model fail, and is the failure caused by the model, by the
   representation, or by the quality of the annotations?

### What a successful solution looks like

A quantitative bar and a qualitative one, both required:

- **Quantitative:** Dice and IoU on the held-out test split that clearly exceed a
  tuned classical baseline. Pixel accuracy is explicitly *not* a success criterion,
  and Chapter 11 shows why.
- **Qualitative:** the predicted mask visibly traces the crack in a four-panel
  comparison (image, ground truth, prediction, overlay), including on hard cases -
  thin cracks, low contrast, shadows, stains and rough texture.

---
# 2. Motivation and importance

### Why this problem matters

Infrastructure inspection is a large, safety-critical, and largely manual industry.
Bridges, tunnels, dams, parking structures and residential buildings are inspected on
fixed cycles, and the inspection itself is often the most expensive part - it can
require scaffolding, rope access, lane closures or drones.

Automating the *visual* part of that inspection changes the economics in three ways:

1. **Scale.** A drone or a phone can capture thousands of images per structure. No
   human can review them all; a model can.
2. **Consistency.** An automated measurement is reproducible. Two inspections six
   months apart become directly comparable, which is what makes **crack propagation
   tracking** possible - and propagation, not presence, is what predicts failure.
3. **Triage.** The model does not need to replace the engineer. Directing expert
   attention to the 2% of images that need it is already a large win.

### Why segmentation rather than classification

This is the central design decision of the project, so it is worth stating plainly.
A classifier answers "is there a crack in this photo?". That is rarely the useful
question - on a structure already known to be cracked, the answer is always yes. The
questions that drive an engineering decision are:

- *Where* is the crack, so it can be revisited and monitored?
- *How long and how wide* is it, so severity can be graded?
- *Has it grown* since the last inspection?

All three require a per-pixel answer. Segmentation produces a mask from which length,
width, orientation and area follow directly by measurement; classification produces a
single bit from which none of them follow.

### Who benefits

Structural engineers and inspection contractors directly; asset owners (municipalities,
transport authorities, facility managers) through cheaper and more frequent
inspection; and researchers in structural health monitoring, who need consistent
large-scale crack measurements that manual annotation cannot supply.

---
# 3. Data collection

### Source

**Ozgenel, C.F. (2019), "Concrete Crack Segmentation Dataset", Mendeley Data, V1.**
DOI: [10.17632/jwsn7tfbrp.1](https://doi.org/10.17632/jwsn7tfbrp.1)

The dataset contains **458 high-resolution photographs** of concrete surfaces on the
METU campus, each paired with a **manually annotated binary crack mask**. Images were
captured with a consumer camera at roughly 1 m from the surface, under varying natural
lighting.

| Folder | Contents |
|---|---|
| `data/concrete_crack_segmentation/rgb/` | Source photographs |
| `data/concrete_crack_segmentation/BW/`  | Manually annotated masks, white = crack |

### Why this dataset and not the popular one

The widely used "Concrete Crack Images for Classification" dataset (40,000 images at
227x227) was **cropped from these same 458 photographs**. It is far larger, but it
ships only a per-image label - crack or no crack - and **no masks**. It cannot train a
segmentation model.

Working from the originals is therefore not a limitation we accepted but the whole
point: we cut our own patches from the full-resolution images *and* their masks using
identical coordinates, which gives us pixel-level supervision that the derived dataset
threw away. It also lets us control the patch size, the sampling policy, and - most
importantly - the train/test split, which the pre-cropped dataset does not.

### Licensing and ethics

The dataset is published under **CC BY 4.0**, which permits use and redistribution
with attribution; the citation above is that attribution. The photographs show
building surfaces only. There are no people, no personally identifying information,
and no privacy concerns. There are no commercial restrictions on academic use.

The dataset is not committed to the repository (683 MB). It is downloaded from the DOI
and extracted under `data/`.

---
# 4. Setup and data acquisition

Imports, the global configuration, and reproducible seeding. Every tunable constant in
this notebook is defined in one place below, so that the final project can scale the
work up by changing `PROPOSAL_SUBSET_N` rather than by hunting through cells.

In [ ]:
# --------------------------------------------------------------- imports
# Import order is deliberate and must not be rearranged. scikit-image and
# scikit-learn link Intel's OpenMP runtime through MKL, and so does PyTorch.
# If torch is imported first, loading skimage afterwards aborts the kernel with
# "OMP: Error #15: Initializing libiomp5md.dll, but found libiomp5md.dll already
# initialized". Importing the MKL-backed libraries before torch loads a single
# OpenMP runtime and avoids the clash without the unsafe KMP_DUPLICATE_LIB_OK
# override.
import copy, json, os, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from skimage.feature import (graycomatrix, graycoprops, hessian_matrix,
                             hessian_matrix_eigvals)
from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import umap

import cv2
import imagehash
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps

import torch                      # must come after the MKL-backed imports above
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore", category=FutureWarning)

# ----------------------------------------------------------------- config
SEED = 42                 # one seed drives every random operation
PATCH = 256               # patch side in pixels, at native resolution
MASK_THRESHOLD = 127      # binarization cut for the lossy JPEG masks
MIN_CRACK_PIXELS = 50     # below this a patch counts as background
PHASH_THRESHOLD = 8       # near-duplicate Hamming distance, justified in ch. 7
SPLIT_FRACTIONS = {"train": 0.70, "val": 0.15, "test": 0.15}
N_STRATA = 4              # crack-ratio quartiles used to stratify the split

# The proposal runs expensive steps on a subset. The final project raises this to
# all 458 images. Every table below states how many images produced it.
PROPOSAL_SUBSET_N = 60

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "figure.autolayout": True})

print(f"python  {os.sys.version.split()[0]}")
print(f"torch   {torch.__version__}  device: {DEVICE}")
if DEVICE == "cuda":
    print(f"gpu     {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print(f"opencv  {cv2.__version__}   pandas {pd.__version__}   numpy {np.__version__}")

### Locating the dataset

The notebook never hardcodes a dataset path and never re-downloads data it already
has. `resolve_data_root` searches the repository for any directory containing both an
`rgb/` and a `BW/` subfolder, so it works regardless of what the extracted archive
folder happens to be called.

In [ ]:
def resolve_data_root(root=Path(".")):
    """Find the extracted dataset: any directory holding both rgb/ and BW/."""
    for cand in sorted(root.rglob("*")):
        if cand.is_dir() and (cand / "rgb").is_dir() and (cand / "BW").is_dir():
            return cand
    raise FileNotFoundError(
        "Dataset not found. Download it from https://doi.org/10.17632/jwsn7tfbrp.1 "
        "and extract it under data/ so that data/<name>/rgb and data/<name>/BW exist.")


DATA_ROOT = resolve_data_root()
n_rgb = len(list((DATA_ROOT / "rgb").iterdir()))
n_bw = len(list((DATA_ROOT / "BW").iterdir()))
print(f"dataset root : {DATA_ROOT.as_posix()}")
print(f"rgb images   : {n_rgb}")
print(f"masks (BW)   : {n_bw}")

### The two loading functions

Everything in this notebook reads images through exactly these two functions. That is
a deliberate constraint, not tidiness: Chapter 6 shows that calling `Image.open`
directly on an RGB file silently produces data that is misaligned with its mask for
194 of the 458 images.

In [ ]:
def load_rgb(path):
    """Open a photograph with EXIF orientation applied.

    PIL stores the orientation tag but does not act on it. The masks were saved with
    rotation already baked in, so skipping exif_transpose here misaligns image and
    mask. See chapter 6 - this single call is the difference between a working
    dataset and a silently broken one.
    """
    with Image.open(path) as im:
        return np.asarray(ImageOps.exif_transpose(im).convert("RGB"))


def load_mask(path, threshold=MASK_THRESHOLD):
    """Open an annotation mask and binarize it. Returns a boolean array, True = crack.

    The masks ship as lossy JPEG, so they are not strictly 0/255 and must be
    thresholded rather than compared to 255. Chapter 6 justifies the threshold.
    """
    with Image.open(path) as im:
        return np.asarray(im.convert("L")) > threshold


def load_small(path, div=8, exif=True):
    """Fast reduced-resolution grayscale decode via the JPEG DCT scaler.

    draft() decodes directly at 1/8 scale inside the JPEG decoder, which is about 8x
    faster than decoding fully and then resizing. Used for whole-dataset passes where
    fine detail is not needed.
    """
    im = Image.open(path)
    im.draft("L", (im.size[0] // div, im.size[1] // div))
    if exif:
        im = ImageOps.exif_transpose(im)
    return np.asarray(im.convert("L"))

---
# 5. Structure of the data

### The three levels

This dataset has an unusual structure for a data science project: the raw unit is an
image, but the modelling unit is a patch, and the prediction unit is a pixel. Being
explicit about all three prevents a lot of confusion later.

| Level | What one item is | How many | Role |
|---|---|---|---|
| **Source image** | One 4032x3024 photograph plus its mask | 458 | The unit the **train/val/test split** is assigned to |
| **Patch** | A 256x256 crop of an image, plus the identically-cropped mask | ~165 per image | The unit the model consumes; one **row** of our metadata table |
| **Pixel** | One position in a patch | 65,536 per patch | The unit being **predicted** and scored |

### The target variable

The target is **per-pixel and binary**: 1 if the pixel was annotated as crack, 0
otherwise. There is no other label in this project. A patch-level "has crack" column
appears in the table below, but it is a *description of the target distribution* used
for sampling and analysis - it is never predicted and never used as a model input.

### The patch metadata table

Rather than writing tens of thousands of cropped JPEGs to disk, we build a table with
**one row per patch** recording where the patch is and what it looks like, and crop
lazily from the source image when the pixels are actually needed. This keeps the
dataset on disk at its original 683 MB instead of several extra gigabytes, and the
table itself is what we run EDA on.

| Column | Type | Meaning |
|---|---|---|
| `image_id` | string | Source photograph, e.g. `046`. The **group key** that prevents leakage |
| `split` | category | `train` / `val` / `test`, inherited from the source image |
| `patch_id`, `y0`, `x0`, `size` | int | Position of the crop in the source image |
| `crack_pixels`, `crack_ratio` | int, float | Target statistics: how much of the patch is crack |
| `has_crack` | bool | `crack_pixels >= 50`. Used for sampling and stratification only |
| `gray_mean`, `gray_std`, `p05`, `p95`, `dyn_range` | float | Brightness and contrast descriptors |
| `edge_density`, `lap_var`, `grad_mean` | float | Edge and focus descriptors |
| `dark_frac` | float | Fraction of pixels more than 2 sigma below the patch mean |
| `sat_mean` | float | Mean HSV saturation - stains are chromatic, cracks are not |
| `glcm_contrast`, `glcm_homogeneity` | float | Texture descriptors from the grey-level co-occurrence matrix |

The descriptor columns exist to answer questions like "does the model fail on
low-contrast patches?" and to drive the clustering in Chapter 9. **They are not model
inputs.** The segmentation model sees raw pixels only.

---
# 6. Data cleaning and quality control

This chapter is longer than a cleaning chapter usually deserves, because this dataset
contains a defect that is invisible to the checks people normally run, and that would
have quietly destroyed the project.

We check, in order:

1. Do the image and mask filenames pair up?
2. Do paired files have the same dimensions?
3. **Are they actually aligned?** (Not the same question as 2.)
4. Are the masks really binary?
5. Are there duplicates?

### 6.1 Pairing images to masks

The two folders do not use the same filename convention. Image IDs are non-contiguous
and run past 600, so pairing by sort order or by index would silently mismatch pairs.
We pair on the **filename stem**, and resolve the extension case-insensitively.

In [ ]:
EXIF_ORIENTATION_TAG = 274

rgb_by_stem = {p.stem: p for p in (DATA_ROOT / "rgb").iterdir() if p.is_file()}
bw_by_stem = {p.stem: p for p in (DATA_ROOT / "BW").iterdir() if p.is_file()}

only_rgb = sorted(set(rgb_by_stem) - set(bw_by_stem))
only_bw = sorted(set(bw_by_stem) - set(rgb_by_stem))
paired = sorted(set(rgb_by_stem) & set(bw_by_stem), key=lambda s: (len(s), s))

print(f"images        : {len(rgb_by_stem)}")
print(f"masks         : {len(bw_by_stem)}")
print(f"matched pairs : {len(paired)}")
print(f"image without mask : {len(only_rgb)}  {only_rgb[:5]}")
print(f"mask without image : {len(only_bw)}  {only_bw[:5]}")

ext_rgb = pd.Series([rgb_by_stem[s].suffix for s in paired]).value_counts()
ext_bw = pd.Series([bw_by_stem[s].suffix for s in paired]).value_counts()
print(f"\nrgb extensions: {ext_rgb.to_dict()}")
print(f"BW  extensions: {ext_bw.to_dict()}")

**Result.** All 458 images pair with a mask and there are no orphans on either side.

Note the extension column: the `rgb/` folder mixes **257 `.jpg` with 201 `.JPG`**
while `BW/` is uniformly lowercase. On Windows this is harmless, but the same code on
Linux or macOS with a case-sensitive filesystem would fail to find 201 of the masks if
we built the mask path by string substitution on the image path. Pairing on the stem
avoids the problem entirely.

### 6.2 Dimensions - and why matching dimensions prove nothing

Now we check that each image has the same dimensions as its mask. We compute the size
two ways: as stored on disk, and after applying the EXIF orientation tag.

**EXIF orientation** is metadata a camera writes when the photo was taken with the
device rotated. It says "this image should be displayed rotated by 90/180/270
degrees". `PIL.Image.open` reports the *stored* dimensions and does not apply the tag.
The annotation tool used to produce the masks evidently did apply it, and saved the
masks already rotated.

In [ ]:
rows = []
for stem in paired:
    rp, bp = rgb_by_stem[stem], bw_by_stem[stem]
    with Image.open(rp) as im:
        raw_w, raw_h = im.size
        orient = im.getexif().get(EXIF_ORIENTATION_TAG)
        fixed_w, fixed_h = ImageOps.exif_transpose(im).size
    with Image.open(bp) as im:
        mask_w, mask_h = im.size
    rows.append(dict(
        image_id=stem, rgb_path=rp.as_posix(), mask_path=bp.as_posix(),
        exif_orientation=orient,
        raw_w=raw_w, raw_h=raw_h, fixed_w=fixed_w, fixed_h=fixed_h,
        mask_w=mask_w, mask_h=mask_h,
        match_raw=(raw_w, raw_h) == (mask_w, mask_h),
        match_fixed=(fixed_w, fixed_h) == (mask_w, mask_h)))

images = pd.DataFrame(rows)

print("EXIF orientation tag:")
print(images.exif_orientation.value_counts(dropna=False).rename(
    {1: "1 = normal", 3: "3 = rotate 180", 6: "6 = rotate 90 CW"}).to_string())
print(f"\ndimensions match BEFORE exif_transpose : "
      f"{images.match_raw.sum()}/{len(images)}")
print(f"dimensions match AFTER  exif_transpose : "
      f"{images.match_fixed.sum()}/{len(images)}")
print("\nresolutions after correction:")
print(images.groupby(['fixed_w', 'fixed_h']).size().sort_values(
    ascending=False).to_string())

**Result, and the trap.**

Before correction only **279 of 458** pairs have matching dimensions. After applying
`exif_transpose`, **all 458 match**. The 179 failures are exactly the images tagged
Orientation = 6 (rotate 90), where the stored image is 4032x3024 but the mask is
3024x4032.

That much a shape assertion would have caught. Here is what it would not:

**15 images are tagged Orientation = 3, meaning rotate 180.** A 180-degree rotation
does not change width or height. Those 15 pass every dimension check while the image
is *upside-down relative to its mask*. Training on them would feed the network 15
images whose labels point at the wrong pixels, and nothing in a conventional pipeline
would report an error.

The lesson generalises: **matching dimensions are not evidence of alignment.** We need
a check based on content.

### 6.3 Verifying alignment by content

We need a test that looks at the pixels. The physical fact we can exploit is simple:
**cracks are darker than the surrounding concrete.** So if a mask is correctly aligned
to its image, the pixels it marks should be measurably darker than the pixels it does
not.

Define an alignment score:

```
score = mean(darkness inside the mask) - mean(darkness outside the mask)
```

where `darkness = 255 - grayscale`. A correctly aligned pair gives a clearly positive
score. We compute it for each image against its mask, and against the same mask
rotated 180 degrees. If our loading is correct, the true orientation should win every
time. This runs on all 458 images at 1/8 resolution.

In [ ]:
def alignment_score(gray, mask):
    """Mean darkness inside the annotated crack minus mean darkness outside."""
    if mask.sum() == 0 or mask.all():
        return np.nan
    dark = 255.0 - gray.astype(np.float32)
    return float(dark[mask].mean() - dark[~mask].mean())


t0 = time.time()
fixed, unfixed, flipped = [], [], []
for r in images.itertuples():
    m = load_small(r.mask_path, exif=False) > MASK_THRESHOLD
    g_fix = load_small(r.rgb_path, exif=True)     # correct loading
    g_raw = load_small(r.rgb_path, exif=False)    # naive loading

    h, w = min(g_fix.shape[0], m.shape[0]), min(g_fix.shape[1], m.shape[1])
    g, mm = g_fix[:h, :w], m[:h, :w]

    fixed.append(alignment_score(g, mm))
    flipped.append(alignment_score(g, mm[::-1, ::-1]))
    unfixed.append(alignment_score(g_raw, m) if g_raw.shape == m.shape else np.nan)

images["align_fixed"] = fixed
images["align_flipped"] = flipped
images["align_unfixed"] = unfixed

print(f"computed in {time.time()-t0:.1f}s\n")
print(f"correct orientation beats 180-rotated : "
      f"{int((images.align_fixed > images.align_flipped).sum())}/{len(images)}")
print(f"mean score, correct orientation : {images.align_fixed.mean():6.2f}")
print(f"mean score, 180-rotated         : {images.align_flipped.mean():6.2f}")

o3 = images[images.exif_orientation == 3]
print(f"\nThe {len(o3)} Orientation=3 images - dimension check cannot see the problem:")
print(f"  dimensions matched without the fix : {o3.match_raw.sum()}/{len(o3)}")
print(f"  alignment score WITH exif fix      : {o3.align_fixed.mean():6.2f}")
print(f"  alignment score WITHOUT exif fix   : {o3.align_unfixed.mean():6.2f}")

**Result.** The correctly loaded orientation beats the 180-rotated alternative on
**458 of 458** images, with mean scores of **68.1 versus 4.9**. The separation is
large and unambiguous, which confirms both that our loading is right and that the
score is a meaningful test rather than noise.

The last three lines are the payoff. For the 15 Orientation=3 images, dimensions
matched **15/15 even without the fix** - so a shape assertion would have passed
silently. But their alignment score is **70.5 with the EXIF fix and 6.0 without**: the
naive loading produces masks that mark pixels no darker than average, which is exactly
what an upside-down mask looks like. This is the check that catches the defect.

**Decision:** every image in this project is loaded through `load_rgb`, which applies
`exif_transpose` unconditionally. We keep this alignment score as a permanent
regression test.

### 6.4 Are the masks actually binary?

The masks are described as binary, but they are stored as **JPEG**, which is a lossy
format. Lossy compression of a hard black-and-white edge produces ringing - a halo of
intermediate grey values around every boundary. Since a crack is almost entirely
boundary, this matters more here than it usually would.

We accumulate the exact value histogram over all 458 masks at full resolution.

In [ ]:
t0 = time.time()
hist = np.zeros(256, dtype=np.int64)
crack_px, total_px = [], []
for path in images.mask_path:
    with Image.open(path) as im:
        m = np.asarray(im.convert("L"))
    hist += np.bincount(m.ravel(), minlength=256)
    crack_px.append(int((m > MASK_THRESHOLD).sum()))
    total_px.append(int(m.size))

images["crack_pixels"] = crack_px
images["total_pixels"] = total_px
images["crack_ratio"] = images.crack_pixels / images.total_pixels

n = hist.sum()
print(f"scanned {len(images)} masks, {n:,} pixels, in {time.time()-t0:.1f}s\n")
print(f"value == 0     (pure black) : {hist[0]:>13,}  {100*hist[0]/n:7.4f}%")
print(f"value == 255   (pure white) : {hist[255]:>13,}  {100*hist[255]/n:7.4f}%")
print(f"value 1..254   (in between) : {hist[1:255].sum():>13,}  "
      f"{100*hist[1:255].sum()/n:7.4f}%")
print(f"\nratio of ambiguous to crack pixels: "
      f"1 : {hist[255]/hist[1:255].sum():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

axes[0].bar(range(256), hist, width=1.0, color="#4C72B0")
axes[0].set_yscale("log")
axes[0].set_title("Mask values are bimodal but not binary")
axes[0].set_xlabel("pixel value"); axes[0].set_ylabel("count (log)")
axes[0].axvline(MASK_THRESHOLD, color="crimson", ls="--",
                label=f"threshold = {MASK_THRESHOLD}")
axes[0].legend()

axes[1].bar(range(1, 255), hist[1:255], width=1.0, color="#DD8452")
axes[1].set_title("Intermediate values only: JPEG ringing around crack edges")
axes[1].set_xlabel("pixel value"); axes[1].set_ylabel("count")
axes[1].axvline(MASK_THRESHOLD, color="crimson", ls="--")
plt.show()

**Result and decision.** The masks are strongly bimodal, as intended, but **0.519% of
all pixels fall strictly between 1 and 254** while only **1.104% are pure white**.
There is roughly **one ambiguous pixel for every two crack pixels** - a far higher
proportion than the "negligible compression noise" one might assume, and a direct
consequence of cracks being thin (nearly all of their pixels are near an edge).

The right panel shows those intermediate values are spread fairly evenly rather than
clustered near 0 or 255, which is characteristic of ringing rather than of genuine
soft labels.

**We binarize at 127**, the midpoint. Two reasons: it is the natural cut for a
symmetric ringing distribution, and the histogram shows no meaningful structure near
it to suggest a better choice. This assigns roughly half the ambiguous pixels to each
class. We record the decision here because it is a real modelling choice: a threshold
of 40 or 200 would visibly thicken or thin every crack in the ground truth, and would
change every Dice score in this notebook.

We also confirm **crack = white**: pure white is the rare value (1.1%) and pure black
the common one (98.4%), and the alignment score in 6.3 was positive, which only holds
if the marked pixels are the dark ones in the photograph.

### 6.5 Crack coverage per image

Before splitting, we need to know how much crack there is per image - both to
understand the class imbalance we are up against and to stratify the split.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

axes[0].hist(100 * images.crack_ratio, bins=50, color="#4C72B0")
axes[0].set_title(f"Crack coverage per image (n={len(images)})")
axes[0].set_xlabel("crack pixels as % of image"); axes[0].set_ylabel("images")
axes[0].axvline(100 * images.crack_ratio.median(), color="crimson", ls="--",
                label=f"median {100*images.crack_ratio.median():.2f}%")
axes[0].legend()

axes[1].hist(images.crack_pixels / 1e3, bins=50, color="#55A868")
axes[1].set_title("Absolute crack size varies over an order of magnitude")
axes[1].set_xlabel("crack pixels (thousands)"); axes[1].set_ylabel("images")
plt.show()

print(f"crack ratio  mean {100*images.crack_ratio.mean():.3f}%  "
      f"median {100*images.crack_ratio.median():.3f}%  "
      f"min {100*images.crack_ratio.min():.3f}%  "
      f"max {100*images.crack_ratio.max():.3f}%")
print(f"images with an entirely empty mask: {(images.crack_pixels == 0).sum()}")

**Result.** Crack coverage averages **1.36%** of an image (median 1.17%), ranging from
0.39% to 8.67%. Two consequences follow, and they shape the rest of the project:

1. **The class imbalance is roughly 1:73.** Any metric that rewards predicting the
   majority class is useless here. Chapter 11 makes this concrete by showing that
   predicting "no crack anywhere" scores 98.6% pixel accuracy.
2. **No image has an empty mask.** Every one of the 458 photographs contains at least
   one crack. This matters for how we interpret the patch data later: all crack-free
   patches in this project are produced by *our cropping*, not present in the source
   collection. The dataset therefore contains no examples of a clean concrete wall
   photographed as such, which is a genuine limitation we return to in Chapter 15.

---
# 7. Preventing data leakage, and the split

### The leakage risk specific to this project

Our modelling unit is a 256x256 patch, but patches are not independent. Two patches
cut from the same photograph share lighting, camera, concrete mix, surface texture and
often the same physical crack. If patches from one image landed in both train and
test, the model could score well by recognising the *wall*, not by learning what a
crack looks like - and we would not be able to tell.

So the split is assigned at the level of the **source image**, before a single patch
is cut. Every patch inherits the split of its parent image and carries `image_id` so
the guarantee can be re-verified at any point.

### A second, subtler risk

Splitting per image is necessary but may not be sufficient. These are sequential
photographs taken on one campus, and several may show the same wall from slightly
different angles. Two near-duplicate photographs in different splits are almost as
bad as the same photograph in both.

So we detect near-duplicates and assign whole duplicate **groups** to a split.

### 7.1 Choosing a near-duplicate threshold instead of copying one

A perceptual hash maps an image to a short bit string such that visually similar
images get similar strings. The usual recipe is "call images duplicates if the Hamming
distance is below about 10". That recipe is dangerous here: concrete surfaces are
visually self-similar by nature, so a loose threshold can decide the entire dataset is
one giant duplicate group.

We therefore compare two hash functions and inspect what each threshold actually does
to the group structure before choosing.

In [ ]:
def phash_bits(paths, fn=imagehash.phash):
    return np.array([fn(Image.fromarray(load_small(p))).hash.ravel() for p in paths])


def hamming_matrix(bits):
    b = bits.astype(np.int8)
    return (b[:, None, :] != b[None, :, :]).sum(-1)


def group_by_distance(dist, threshold):
    """Union-find over all pairs within threshold. Returns a group id per image."""
    n = len(dist)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    ii, jj = np.where(np.triu(dist <= threshold, k=1))
    for a, b in zip(ii, jj):
        ra, rb = find(int(a)), find(int(b))
        if ra != rb:
            parent[rb] = ra
    roots = [find(i) for i in range(n)]
    remap = {r: g for g, r in enumerate(sorted(set(roots)))}
    return np.array([remap[r] for r in roots])


probe = []
for name, fn in [("phash", imagehash.phash), ("dhash", imagehash.dhash)]:
    dist = hamming_matrix(phash_bits(images.rgb_path, fn))
    for thr in [0, 4, 8, 12]:
        sizes = pd.Series(group_by_distance(dist, thr)).value_counts()
        probe.append(dict(hash=name, threshold=thr, groups=sizes.size,
                          largest_group=sizes.max(),
                          images_in_multi=int(sizes[sizes > 1].sum())))
    if name == "phash":
        phash_dist = dist

print(pd.DataFrame(probe).to_string(index=False))

**Result, and why the choice matters.** The two hashes behave completely differently:

- **phash is stable.** Even at distance <= 12 the largest group holds 3 images. Its
  pairwise distance distribution is well separated, so a threshold of 8 sits far out
  in the tail.
- **dhash collapses.** At distance <= 12 it merges **296 of 458 images** into a single
  component - 65% of the dataset declared mutually duplicate. dhash compares adjacent
  pixel gradients, which for large flat expanses of grey concrete are nearly identical
  regardless of which wall is photographed.

Had we accepted the folk threshold with dhash, our "leakage-safe" grouping would have
forced two thirds of the data into one split and destroyed the experiment.

**We use phash with a Hamming distance of 8.**

The honest conclusion is also worth stating: **near-duplication in this dataset turns
out to be minimal** - about 8 duplicate pairs covering 16 images out of 458. The
grouping is cheap insurance rather than a major correction, and we report its true
size rather than overselling it.

In [ ]:
images["dup_group"] = group_by_distance(phash_dist, PHASH_THRESHOLD)
sizes = images.dup_group.value_counts()

print(f"phash threshold <= {PHASH_THRESHOLD}")
print(f"  groups                       : {sizes.size}")
print(f"  largest group                : {sizes.max()} images")
print(f"  groups with more than 1 image: {(sizes > 1).sum()}")
print(f"  images inside such groups    : {sizes[sizes > 1].sum()}")

dups = images[images.dup_group.isin(sizes[sizes > 1].index)].sort_values("dup_group")
if len(dups):
    fig, axes = plt.subplots(2, 4, figsize=(13, 6))
    for ax, r in zip(axes.ravel(), dups.head(8).itertuples()):
        ax.imshow(load_small(r.rgb_path), cmap="gray")
        ax.set_title(f"id {r.image_id}  group {r.dup_group}", fontsize=9)
        ax.axis("off")
    fig.suptitle("Detected near-duplicate pairs (adjacent titles sharing a group id)")
    plt.show()

**Interpretation.** The detected pairs are genuine: images sharing a group id show the
same wall section, typically two consecutive shots. They are visually near-identical
but not byte-identical, which is exactly the case a file-hash check would miss and a
perceptual hash catches.

### 7.2 Building the split

We assign whole duplicate groups to train / validation / test in a 70 / 15 / 15 split,
**stratified by crack coverage**. Stratification matters because crack coverage spans
0.39% to 8.67%: an unstratified random draw could easily give the test split a
disproportionate share of the heavily-cracked images and make the model look better or
worse than it is.

The split is written to `outputs/split.csv` and **frozen**. The final report loads
that file rather than recomputing, so the leakage guarantees established here carry
forward unchanged.

In [ ]:
def stratified_group_split(frame, fractions, seed=SEED):
    """Assign whole duplicate groups to splits, balancing crack-ratio strata.

    Greedy allocation: within each stratum, walk the groups in shuffled order and give
    each to whichever split is currently furthest below its target share of images.
    """
    rng = np.random.default_rng(seed)
    names = list(fractions)
    assigned = {n: 0 for n in names}
    out = {}
    sizes = frame.groupby("dup_group").size()
    group_stratum = frame.groupby("dup_group").stratum.first()

    for stratum in sorted(frame.stratum.unique()):
        gids = rng.permutation(group_stratum[group_stratum == stratum].index.values)
        for gid in gids:
            total = max(sum(assigned.values()), 1)
            pick = max(names, key=lambda n: fractions[n] - assigned[n] / total)
            out[gid] = pick
            assigned[pick] += int(sizes[gid])
    return out


images["stratum"] = pd.qcut(images.crack_ratio, N_STRATA, labels=False)
images["split"] = images.dup_group.map(
    stratified_group_split(images, SPLIT_FRACTIONS))

summary = images.groupby("split").agg(
    images=("image_id", "size"),
    pct=("image_id", lambda s: round(100 * len(s) / len(images), 1)),
    crack_ratio_mean=("crack_ratio", lambda s: round(100 * s.mean(), 3)),
    crack_ratio_median=("crack_ratio", lambda s: round(100 * s.median(), 3)),
).reindex(["train", "val", "test"])
print(summary.to_string())

### 7.3 Verifying that the split is actually leak-free

An assertion that encodes a methodological claim is evidence, so it stays in the
notebook and prints its result rather than passing silently.

In [ ]:
sets = {n: set(images.loc[images.split == n, "image_id"]) for n in SPLIT_FRACTIONS}
gsets = {n: set(images.loc[images.split == n, "dup_group"]) for n in SPLIT_FRACTIONS}

for a in sets:
    for b in sets:
        if a < b:
            assert not (sets[a] & sets[b]), f"image id overlap: {a} and {b}"
            assert not (gsets[a] & gsets[b]), f"duplicate group overlap: {a} and {b}"
assert sum(len(s) for s in sets.values()) == len(images), "images lost or duplicated"

print("PASS  no image_id appears in more than one split")
print("PASS  no duplicate group is split across two splits")
print(f"PASS  all {len(images)} images assigned exactly once")

cols = ["image_id", "split", "dup_group", "stratum", "rgb_path", "mask_path",
        "fixed_w", "fixed_h", "exif_orientation", "crack_pixels", "crack_ratio"]
images[cols].to_csv(OUT / "split.csv", index=False)
print(f"\nfrozen split written to {(OUT / 'split.csv').as_posix()}")

**Result.** The split is **320 / 69 / 69 images** (69.9 / 15.1 / 15.1%), with mean
crack coverage of 1.39 / 1.26 / 1.34% - well balanced, which confirms the
stratification worked. All three assertions pass.

From this point on, **the test split is not touched again in this notebook.** Every
number reported below comes from train or validation. The test split is reserved for
the final report, so that it remains a genuine held-out estimate rather than a set we
have implicitly tuned against.

---
# 8. Patch generation

### Why crop instead of resize

A 4032x3024 photograph cannot be fed to a network at full resolution. The obvious
alternative - resize the whole image to 256x256 - **destroys the target**. Cracks are
a few pixels wide at native resolution; a 16x downscale reduces them to a fraction of
a pixel, and the interpolation averages them into the surrounding concrete. The
annotation would survive as a thin grey smear that no longer corresponds to anything
visible in the image.

So we crop at native resolution. Each patch covers a small physical area but preserves
the fine detail that makes a crack identifiable.

### The alignment guarantee

Image and mask are cropped with **the same coordinates in the same call**, so
misalignment is structurally impossible rather than merely unlikely. Combined with the
EXIF handling from Chapter 6, this is the second half of the alignment guarantee.

### Grid geometry

We use a non-overlapping grid with stride equal to the patch size. Images are not all
the same shape, and both orientations occur, so the grid is computed from each image's
actual dimensions. Remainder strips at the right and bottom edge (less than 256 px)
are dropped rather than padded - padding would introduce artificial content, and the
loss is under 6% of each image.

In [ ]:
def grid_origins(h, w, patch=PATCH):
    """Top-left corners of a non-overlapping grid. Remainder edges are dropped."""
    return [(y, x) for y in range(0, h - patch + 1, patch)
            for x in range(0, w - patch + 1, patch)]


def patch_descriptors(rgb_patch):
    """Appearance descriptors for one patch. EDA and error analysis only -
    these are never inputs to the segmentation model."""
    gray = cv2.cvtColor(rgb_patch, cv2.COLOR_RGB2GRAY)
    g = gray.astype(np.float32)
    p05, p95 = np.percentile(g, [5, 95])
    gx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    mean, std = float(g.mean()), float(g.std())

    q = (cv2.resize(gray, (64, 64), interpolation=cv2.INTER_AREA) // 32).astype(np.uint8)
    glcm = graycomatrix(q, distances=[1], angles=[0, np.pi / 2], levels=8,
                        symmetric=True, normed=True)

    return dict(
        gray_mean=mean, gray_std=std, p05=float(p05), p95=float(p95),
        dyn_range=float(p95 - p05),
        edge_density=float((cv2.Canny(gray, 50, 150) > 0).mean()),
        lap_var=float(cv2.Laplacian(gray, cv2.CV_32F).var()),
        grad_mean=float(np.sqrt(gx * gx + gy * gy).mean()),
        dark_frac=float((g < mean - 2 * std).mean()) if std > 0 else 0.0,
        sat_mean=float(cv2.cvtColor(rgb_patch, cv2.COLOR_RGB2HSV)[:, :, 1].mean()),
        glcm_contrast=float(graycoprops(glcm, "contrast").mean()),
        glcm_homogeneity=float(graycoprops(glcm, "homogeneity").mean()))


def patches_for_image(row, patch=PATCH):
    rgb, mask = load_rgb(row.rgb_path), load_mask(row.mask_path)
    assert rgb.shape[:2] == mask.shape, f"{row.image_id}: shape mismatch"
    out = []
    for k, (y, x) in enumerate(grid_origins(*mask.shape, patch=patch)):
        rp = rgb[y:y + patch, x:x + patch]          # identical coordinates
        mp = mask[y:y + patch, x:x + patch]         # for image and mask
        cp = int(mp.sum())
        rec = dict(image_id=row.image_id, split=row.split, patch_id=k,
                   y0=y, x0=x, size=patch, crack_pixels=cp,
                   crack_ratio=cp / (patch * patch),
                   has_crack=cp >= MIN_CRACK_PIXELS)
        rec.update(patch_descriptors(rp))
        out.append(rec)
    return out

### Choosing the subset

Building the metadata table for all 458 images would take about 4.5 minutes and
produce ~75,600 rows. For the proposal we use **60 images**, drawn proportionally from
all three splits so every split is represented. `PROPOSAL_SUBSET_N` at the top of the
notebook is the only thing the final project needs to change.

In [ ]:
def choose_subset(frame, n, seed=SEED):
    rng = np.random.default_rng(seed)
    picks = []
    for name, frac in SPLIT_FRACTIONS.items():
        pool = frame.loc[frame.split == name, "image_id"].values
        picks.extend(rng.choice(pool, size=min(max(1, round(n * frac)), len(pool)),
                                replace=False))
    return frame[frame.image_id.isin(picks)].reset_index(drop=True)


subset = choose_subset(images, PROPOSAL_SUBSET_N)
print(f"subset: {len(subset)} of {len(images)} images "
      f"{subset.split.value_counts().reindex(['train','val','test']).to_dict()}")

t0 = time.time()
rows = []
for r in subset.itertuples():
    rows.extend(patches_for_image(r))
patches = pd.DataFrame(rows)
patches.to_parquet(OUT / "patch_index.parquet", index=False)

print(f"{len(patches):,} patches in {time.time()-t0:.1f}s "
      f"({len(patches)/len(subset):.0f} per image)")
patches.head(3)

In [ ]:
per_split = patches.groupby("split").agg(
    patches=("patch_id", "size"),
    with_crack=("has_crack", "sum"),
    pct_with_crack=("has_crack", lambda s: round(100 * s.mean(), 1)),
).reindex(["train", "val", "test"])
print(per_split.to_string())

empty = (patches.crack_pixels == 0).mean()
print(f"\ncompletely empty patches      : {100*empty:.1f}%")
print(f"1..{MIN_CRACK_PIXELS-1} crack pixels (near-empty) : "
      f"{100*((patches.crack_pixels > 0) & (~patches.has_crack)).mean():.1f}%")
print(f"pixel-level crack rate        : {100*patches.crack_ratio.mean():.3f}%")

**Result.** 60 images produce **9,672 patches**, about 161 per image. Roughly **16%
contain a crack and 83.4% are completely empty**, and this proportion is consistent
across the three splits (16.3 / 15.6 / 16.1%), which is a useful confirmation that the
stratified split did not accidentally concentrate cracks anywhere.

The empty-patch majority is the single most important property of this dataset for
modelling. It drives the loss function (Chapter 10), the training sampling policy
below, and every metric decision in Chapter 11.

**Sampling policy.** For *training* we will rebalance - keeping all crack-bearing
patches and an equal number of randomly chosen empty ones - because a model trained on
84% empty patches learns to output empty masks. For *validation and test* we keep the
complete, unsampled grid. Rebalancing an evaluation set would inflate every score we
report.

### Verifying alignment visually

The quantitative check in Chapter 6 said alignment is correct. This is the check a
human can actually confirm: the ground-truth contour drawn on top of the image patch.
If cropping were misaligned by even a few pixels, the red outline would sit beside the
crack rather than on it.

In [ ]:
show = patches[patches.has_crack].sample(4, random_state=SEED)
paths = images.set_index("image_id")[["rgb_path", "mask_path"]]

fig, axes = plt.subplots(3, 4, figsize=(13, 9.5))
for col, r in enumerate(show.itertuples()):
    p = paths.loc[r.image_id]
    rgb = load_rgb(p.rgb_path)[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size]
    msk = load_mask(p.mask_path)[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size]

    axes[0, col].imshow(rgb)
    axes[0, col].set_title(f"image {r.image_id} patch {r.patch_id}", fontsize=9)
    axes[1, col].imshow(msk, cmap="gray")
    axes[1, col].set_title(f"mask: {r.crack_pixels:,} px "
                           f"({100*r.crack_ratio:.2f}%)", fontsize=9)
    axes[2, col].imshow(rgb)
    axes[2, col].contour(msk.astype(float), levels=[0.5], colors="red", linewidths=1.2)
    axes[2, col].set_title("ground truth contour on image", fontsize=9)
    for row in range(3):
        axes[row, col].axis("off")
fig.suptitle("Aligned image / mask patches - the contour must lie on the crack",
             fontsize=12)
plt.show()

**Interpretation.** The red contours trace the visible cracks in every panel. This
confirms end to end that EXIF handling, mask binarization and identical-coordinate
cropping all work together correctly.

It also shows something about **annotation quality** that will matter in Chapter 12:
the manual masks are somewhat *thicker* than the visible crack and smooth over fine
branching. Some of what we will later score as false positives at crack boundaries is
really annotator imprecision, not model error.

---
# 9. Exploratory data analysis

The patch metadata table is a conventional tabular dataset - 9,672 rows, 12 numeric
descriptors - so ordinary EDA tooling applies. The question we want it to answer is
not "what do cracks look like" but:

> **What visually distinct kinds of patch does this dataset contain, and will the
> model be equally good at all of them?**

That framing is what makes the clustering in 9.4 useful rather than decorative: the
clusters become the **strata we report segmentation performance over** in Chapter 12.

### 9.1 The target distribution across patches

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

axes[0].hist(patches.crack_ratio * 100, bins=60, color="#4C72B0")
axes[0].set_yscale("log")
axes[0].set_title("Most patches contain no crack at all")
axes[0].set_xlabel("crack pixels as % of patch"); axes[0].set_ylabel("patches (log)")

cracked = patches[patches.has_crack]
axes[1].hist(cracked.crack_ratio * 100, bins=50, color="#55A868")
axes[1].set_title(f"Among crack-bearing patches only (n={len(cracked):,})")
axes[1].set_xlabel("crack pixels as % of patch"); axes[1].set_ylabel("patches")

frac = patches.groupby("image_id").has_crack.mean() * 100
axes[2].hist(frac, bins=25, color="#C44E52")
axes[2].set_title("Share of an image's patches that contain crack")
axes[2].set_xlabel("% of patches with crack"); axes[2].set_ylabel("images")
plt.show()

print(f"crack-bearing patches: {len(cracked):,} of {len(patches):,} "
      f"({100*len(cracked)/len(patches):.1f}%)")
print(f"within those, crack covers mean {100*cracked.crack_ratio.mean():.2f}% "
      f"median {100*cracked.crack_ratio.median():.2f}% of the patch")

**Interpretation.** The left panel needs a log axis to be readable at all, which is
itself the finding: the target distribution is dominated by a spike at zero. The
middle panel shows that even *within* crack-bearing patches the crack occupies only a
few percent of pixels - so the imbalance is severe at both levels, between patches and
within them.

The right panel shows the per-image spread: in some images barely 5% of patches
contain crack, in others over 40%. A crack is a connected curve crossing an image, so
the number of patches it touches depends on its length and how diagonally it runs.
This is why the split had to be stratified by crack coverage.

### 9.2 What the descriptors measure, and how they relate

Before reducing dimensions we check whether the twelve descriptors are actually
measuring twelve different things. Highly correlated descriptors would make PCA
misleading and clustering unstable.

In [ ]:
FEATURES = ["gray_mean", "gray_std", "p05", "p95", "dyn_range", "edge_density",
            "lap_var", "grad_mean", "dark_frac", "sat_mean",
            "glcm_contrast", "glcm_homogeneity"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(patches[FEATURES].corr(), cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7}, ax=axes[0], cbar=False)
axes[0].set_title("Descriptor correlation")

corr_target = (patches[FEATURES + ["crack_ratio"]].corr()["crack_ratio"]
               .drop("crack_ratio").sort_values())
corr_target.plot.barh(ax=axes[1], color=np.where(corr_target > 0, "#C44E52", "#4C72B0"))
axes[1].set_title("Correlation of each descriptor with crack coverage")
axes[1].set_xlabel("Pearson r")
plt.show()

**Interpretation.** Two blocks of redundancy are visible. `gray_mean`, `p05` and `p95`
move together (they all measure brightness), and `gray_std`, `dyn_range`, `lap_var`,
`grad_mean` and `glcm_contrast` form a second block measuring local variation.
`glcm_homogeneity` is strongly negatively correlated with `glcm_contrast`, as expected
since they are two summaries of the same co-occurrence matrix.

The right panel is the more interesting one. The descriptors most correlated with
crack coverage are the **texture and edge** measures, not brightness - which is
encouraging, because it means a crack is not simply "a dark region" that a threshold
could find. But the correlations are moderate, not near 1. **No single hand-crafted
descriptor separates cracked from clean patches**, which is precisely the argument for
learning features rather than designing them, and sets up the baseline comparison in
Chapter 12.

### 9.3 Dimensionality reduction - PCA, t-SNE and UMAP compared

We compare three projections rather than picking one, because they answer different
questions: PCA preserves global variance and is linear and deterministic; t-SNE and
UMAP preserve local neighbourhood structure and can reveal clusters PCA cannot.

Hyperparameters are chosen deliberately, not left at defaults:

- **t-SNE `perplexity=30`** - roughly the number of neighbours each point is fitted
  to. With several thousand points, 30 is the standard middle of the useful 5-50 range;
  much lower fragments the map into noise, much higher blurs local structure away.
- **UMAP `n_neighbors=30`** matched to the t-SNE perplexity so the two are comparable,
  and **`min_dist=0.1`**, low enough to let tight groups stay tight.
- Both are run on **standardised** features, since the descriptors differ by orders of
  magnitude, and on a seeded sample of 3,000 patches to keep runtime reasonable.

In [ ]:
EMB_N = 3000
emb_df = patches.sample(min(EMB_N, len(patches)), random_state=SEED).reset_index(drop=True)
X = StandardScaler().fit_transform(emb_df[FEATURES].values)
colour = np.log10(emb_df.crack_ratio.values * 100 + 1e-3)

t0 = time.time()
pca = PCA(n_components=2, random_state=SEED)
E_pca = pca.fit_transform(X)
E_tsne = TSNE(n_components=2, perplexity=30, init="pca",
              random_state=SEED).fit_transform(X)
E_umap = umap.UMAP(n_neighbors=30, min_dist=0.1, n_components=2,
                   random_state=SEED).fit_transform(X)
print(f"three embeddings of {len(X):,} patches in {time.time()-t0:.1f}s")
print(f"PCA explained variance: PC1 {pca.explained_variance_ratio_[0]:.1%}, "
      f"PC2 {pca.explained_variance_ratio_[1]:.1%}, "
      f"together {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
for ax, E, name in [(axes[0], E_pca, "PCA"), (axes[1], E_tsne, "t-SNE"),
                    (axes[2], E_umap, "UMAP")]:
    s = ax.scatter(E[:, 0], E[:, 1], c=colour, s=4, cmap="viridis", alpha=0.7)
    ax.set_title(f"{name} of patch descriptors"); ax.set_xticks([]); ax.set_yticks([])
cb = fig.colorbar(s, ax=axes, fraction=0.02)
cb.set_label("log10(crack % of patch)")
plt.show()

**Interpretation.** The first two principal components capture a substantial share of
the descriptor variance, confirming the redundancy the correlation matrix showed - the
twelve descriptors span far fewer than twelve effective dimensions.

The important observation is about **colour**, which encodes crack coverage. In all
three projections the crack-bearing patches (yellow) are **not cleanly separated**
from empty ones; they form a gradient and overlap heavily with the main mass. This is
a genuinely useful negative result: if crack-bearing patches were separable in this
descriptor space, a simple model on these twelve features would solve the problem, and
Chapter 12 would not need a U-Net. They are not, so it does.

t-SNE and UMAP both reveal filament and cluster structure that PCA smears together.
That structure is not about cracks - it is about *surface appearance*, which is what
we exploit next.

### 9.4 Clustering - discovering the visual regimes

We now ask what those groups are. We compare three clustering algorithms and select
using the **silhouette score**, which measures how much closer a point is to its own
cluster than to the nearest other one (1 is ideal, 0 means overlapping, negative means
misassigned).

In [ ]:
results = []
for k in [2, 3, 4, 5, 6, 7, 8]:
    lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X)
    results.append(dict(method=f"KMeans k={k}", n_clusters=k,
                        silhouette=silhouette_score(X, lab)))
for k in [3, 4, 5]:
    lab = AgglomerativeClustering(n_clusters=k).fit_predict(X)
    results.append(dict(method=f"Agglomerative k={k}", n_clusters=k,
                        silhouette=silhouette_score(X, lab)))
for eps in [1.0, 1.5, 2.0, 2.5]:
    lab = DBSCAN(eps=eps, min_samples=15).fit_predict(X)
    if len(set(lab[lab >= 0])) > 1:
        results.append(dict(method=f"DBSCAN eps={eps}",
                            n_clusters=len(set(lab[lab >= 0])),
                            silhouette=silhouette_score(X[lab >= 0], lab[lab >= 0]),
                            noise_pct=round(100 * (lab == -1).mean(), 1)))

print(pd.DataFrame(results).round(3).to_string(index=False))

In [ ]:
scores = pd.DataFrame([r for r in results if r["method"].startswith("KMeans")])
best_k = int(scores.loc[scores.silhouette.idxmax(), "n_clusters"])

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(scores.n_clusters, scores.silhouette, "o-", color="#4C72B0")
ax.axvline(best_k, color="crimson", ls="--", label=f"chosen k = {best_k}")
ax.set_xlabel("number of clusters"); ax.set_ylabel("silhouette score")
ax.set_title("KMeans cluster count selected by silhouette")
ax.legend(); plt.show()

emb_df["cluster"] = KMeans(n_clusters=best_k, n_init=10,
                           random_state=SEED).fit_predict(X)
profile = emb_df.groupby("cluster").agg(
    patches=("cluster", "size"),
    pct_with_crack=("has_crack", lambda s: round(100 * s.mean(), 1)),
    brightness=("gray_mean", "mean"),
    contrast=("dyn_range", "mean"),
    edges=("edge_density", "mean"),
    texture=("glcm_contrast", "mean"),
    saturation=("sat_mean", "mean")).round(2)
print(profile.to_string())

In [ ]:
fig, axes = plt.subplots(best_k, 5, figsize=(13, 2.5 * best_k))
for c in range(best_k):
    sample = emb_df[emb_df.cluster == c].sample(
        min(5, (emb_df.cluster == c).sum()), random_state=SEED)
    for j, r in enumerate(sample.itertuples()):
        p = paths.loc[r.image_id]
        img = load_rgb(p.rgb_path)[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size]
        axes[c, j].imshow(img); axes[c, j].axis("off")
        if j == 0:
            axes[c, j].set_ylabel(f"cluster {c}")
            axes[c, j].set_title(f"cluster {c}", loc="left", fontsize=10)
    for j in range(len(sample), 5):
        axes[c, j].axis("off")
fig.suptitle("Representative patches per cluster", fontsize=12)
plt.show()

**Interpretation.** The silhouette curve and the cluster profiles together tell us
what the algorithm found. The clusters separate primarily on **brightness and texture
energy** - which, looking at the sample patches, correspond to recognisable physical
regimes: smooth well-lit concrete, rough or aggregate-exposed surface, patches in
shadow, and patches containing high-contrast structure such as edges of the wall,
stains or the cracks themselves.

Crucially, the `pct_with_crack` column shows crack-bearing patches are **not confined
to one cluster** - they appear across several regimes. That is what makes these
clusters useful as evaluation strata: they describe conditions the model must work
under, and they are independent of the label.

**We carry `cluster` forward into Chapter 12** to report Dice per regime, which turns
this analysis into an error-analysis tool rather than an isolated exercise. A single
average Dice hides whether the model works everywhere or only on easy, high-contrast
patches.

---
# 10. Planned models

All models solve the **same task** - RGB patch in, binary mask out - so they are
directly comparable on the same metrics. We build up in three stages of increasing
sophistication, which is deliberate: a deep model is only interesting if it beats a
serious classical attempt, and a serious classical attempt requires actually tuning it.

| | Model | Idea |
|---|---|---|
| **A** | Classical image processing | Cracks are dark, thin, curvilinear. Morphology is designed for exactly that. |
| **B** | Per-pixel classifier on a filter bank | Learn the decision, keep the features hand-designed and interpretable. |
| **C** | U-Net | Learn features and decision jointly, with spatial context. |

### A. Classical image processing

Three variants, since the comparison between them is itself informative:

- **A1 Otsu on inverted grayscale.** The textbook first attempt: cracks are dark, so
  invert and threshold at the value that best separates the two modes.
- **A2 CLAHE + black-hat + per-patch Otsu.** *Black-hat* morphology returns the
  difference between the morphological closing and the image, which isolates dark
  structures **thinner than the structuring element** while cancelling slow brightness
  gradients. That is a much better description of a crack than "dark pixel".
- **A3 the same, with a global threshold learned on train.** Chapter 12 shows why this
  change matters more than the morphology does.

### B. Per-pixel classification on a multiscale filter bank

Every pixel is described by a vector of filter responses at several scales, then
classified independently; the per-pixel decisions reassemble into a mask. This is a
real segmentation method, and it is the "simple, interpretable model" of the project.

The filter bank is written out explicitly so **every feature has a name** - which is
what makes the explainability analysis in Chapter 13 possible at all. Per scale
sigma in {1, 2, 4, 8}: Gaussian smoothing, gradient magnitude, Laplacian of Gaussian,
and both **Hessian eigenvalues**. The Hessian eigenvalues are the theoretically
motivated feature: a crack is a dark curvilinear ridge, and the eigenvalues of the
image Hessian are the standard ridge/valley detector used in vessel and crack
detection. Plus raw grayscale, local standard deviation, and saturation - 23 features.

We fit **LogisticRegression** (linear, fast, interpretable coefficients) and
**RandomForest** (non-linear, gives feature importances).

### C. U-Net

An encoder-decoder convolutional network. The encoder downsamples to build a large
receptive field, so the network can use context to tell a crack from a scratch; the
decoder upsamples back to full resolution. **Skip connections** carry high-resolution
detail from each encoder level directly to the matching decoder level, which is what
makes pixel-accurate boundaries possible - pooling is what gives the network context,
and it is also what destroys the precision needed to outline a two-pixel-wide crack.

**Loss.** Binary cross-entropy alone fails here. With 98.9% background pixels, the
gradient from the majority class dominates and the network converges to predicting
nothing - which, as Chapter 11 shows, scores 98.6% pixel accuracy. Dice loss is
computed from the overlap ratio and is therefore insensitive to how frequent the
positive class is, keeping gradient on the rare crack pixels. We use **BCE + Dice**:
BCE for stable, well-calibrated per-pixel probabilities, Dice for overlap awareness.

**Augmentation** - horizontal and vertical flips and 90-degree rotations, applied to
image and mask identically, **on the training split only**. Cracks have no canonical
orientation, so these transforms produce genuinely plausible new examples.

---
# 11. Evaluation protocol

### The metrics

With `TP`, `FP`, `FN` counted over pixels:

| Metric | Formula | Reads as |
|---|---|---|
| **Dice** | `2*TP / (2*TP + FP + FN)` | Overlap between prediction and truth, harmonic-mean flavoured. Our headline metric. |
| **IoU** | `TP / (TP + FP + FN)` | Intersection over union. Monotonically related to Dice but always lower - never compare one to the other. |
| **Precision** | `TP / (TP + FP)` | Of the pixels called crack, how many were. |
| **Recall** | `TP / (TP + FN)` | Of the actual crack pixels, how many were found. |

Precision and recall are reported alongside Dice because they diagnose *how* a model
fails: over-predicting produces high recall and low precision, timidity does the
reverse, and Dice alone cannot distinguish them.

### Why pixel accuracy is excluded

Crack pixels are about 1.1% of the dataset. A model that outputs an empty mask
everywhere achieves **98.6% pixel accuracy** while finding nothing at all. We compute
this number once, in Chapter 12, purely to demonstrate the point - and then never use
accuracy again.

### The empty-mask problem, and our convention

Dice and IoU are `0/0` when the ground truth is empty and the prediction is empty too,
and **83.4% of our patches have empty ground truth**. This is not an edge case here,
it is the common case, so the convention has to be stated:

> If `TP = FP = FN = 0` - nothing to find and nothing predicted - we score **1.0**. An
> empty prediction on an empty target is exactly correct. Every other degenerate case
> scores 0.

### Micro and macro, and why we report both

- **Micro**: pool TP, FP and FN over every patch, then compute the metric once.
  Each *pixel* carries equal weight.
- **Macro**: compute the metric per patch, then average. Each *patch* carries equal
  weight.

They diverge dramatically in this dataset. Under our convention, the 83.4% empty
patches each score a perfect 1.0, so **macro Dice is dominated by patches where the
correct answer is to do nothing** - a do-nothing model scores 0.838 macro. Micro Dice
for the same model is 0.000.

Neither is wrong; they answer different questions. We therefore report **micro Dice as
the headline**, and additionally report macro restricted to crack-bearing patches,
which measures quality where there is actually something to find. Any Dice number in
this notebook says which one it is.

### Protocol discipline

- Hyperparameters and thresholds are chosen on **train or validation only**.
- Validation and test use the **complete, deterministic, unsampled** patch grid. Only
  training rebalances or augments.
- **The test split is not touched in this proposal at all.** Every number below is
  validation. Test is reserved for the final report.
- Cross-validation uses **GroupKFold grouped by `image_id`**, so no source image
  appears in both a fold's train and validation portions.

In [ ]:
def confusion(pred, true):
    """Pixel counts (tp, fp, fn, tn) for two boolean arrays."""
    tp = int(np.count_nonzero(pred & true))
    fp = int(np.count_nonzero(pred & ~true))
    fn = int(np.count_nonzero(~pred & true))
    return tp, fp, fn, pred.size - tp - fp - fn


def scores_from_counts(tp, fp, fn):
    """Dice, IoU, precision, recall. Empty-on-empty scores a perfect 1.0."""
    if tp + fp + fn == 0:
        return dict(dice=1.0, iou=1.0, precision=1.0, recall=1.0)
    return dict(dice=2 * tp / (2 * tp + fp + fn),
                iou=tp / (tp + fp + fn),
                precision=tp / (tp + fp) if tp + fp else 0.0,
                recall=tp / (tp + fn) if tp + fn else 0.0)


def evaluate(pairs):
    """Aggregate micro and macro metrics over (prediction, truth) patch pairs."""
    TP = FP = FN = TN = 0
    per_patch, per_cracked = [], []
    for pred, true in pairs:
        tp, fp, fn, tn = confusion(pred, true)
        TP += tp; FP += fp; FN += fn; TN += tn
        s = scores_from_counts(tp, fp, fn)
        per_patch.append(s)
        if true.any():
            per_cracked.append(s)

    out = {f"micro_{k}": v for k, v in scores_from_counts(TP, FP, FN).items()}
    for k in ["dice", "iou", "precision", "recall"]:
        out[f"macro_{k}"] = float(np.mean([s[k] for s in per_patch]))
        out[f"cracked_{k}"] = (float(np.mean([s[k] for s in per_cracked]))
                               if per_cracked else float("nan"))
    out["pixel_accuracy"] = (TP + TN) / max(TP + FP + FN + TN, 1)
    out["n_patches"] = len(per_patch)
    out["n_cracked"] = len(per_cracked)
    return out


def stream_patches(patch_df, image_df, split):
    """Yield (rgb, mask) patch pairs, loading each source image only once."""
    sel = patch_df[patch_df.split == split]
    p = image_df.set_index("image_id")[["rgb_path", "mask_path"]]
    for image_id, grp in sel.groupby("image_id"):
        row = p.loc[image_id]
        rgb, mask = load_rgb(row.rgb_path), load_mask(row.mask_path)
        for r in grp.itertuples():
            yield (rgb[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size],
                   mask[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size])


print("metric helpers defined")

---
# 12. Preliminary results

Everything below is computed, not planned. All numbers are on the **validation**
split; the test split remains untouched.

### 12.1 Baseline A - classical image processing

In [ ]:
_clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
MIN_COMPONENT = 60


def predict_nothing(rgb):
    """A0: the do-nothing model, included to expose what accuracy is worth here."""
    return np.zeros(rgb.shape[:2], dtype=bool)


def predict_otsu(rgb):
    """A1: global Otsu on inverted grayscale."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, th = cv2.threshold(255 - gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return th > 0


def blackhat_response(rgb, kernel):
    gray = _clahe.apply(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY))
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel, kernel))
    return cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k)


def clean_components(binary, min_component=MIN_COMPONENT):
    th = cv2.morphologyEx((binary.astype(np.uint8)) * 255, cv2.MORPH_OPEN,
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    n, labels, stats, _ = cv2.connectedComponentsWithStats(th, connectivity=8)
    keep = np.zeros(n, dtype=bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= min_component
    return keep[labels]


def predict_blackhat_otsu(rgb, kernel=15):
    """A2: black-hat, then a per-patch Otsu threshold."""
    bh = blackhat_response(rgb, kernel)
    _, th = cv2.threshold(bh, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return clean_components(th > 0)

Before evaluating A2 we diagnose A1, because its failure explains what the fix has to
be. Otsu chooses a threshold that best separates a histogram into two modes - and it
is **obliged to return a split even when the patch contains only one mode**. On an
empty patch of uniform concrete there is no crack to find, so Otsu splits sensor
noise.

In [ ]:
frac_empty, frac_cracked = [], []
for rgb, m in stream_patches(patches, images, "val"):
    f = predict_otsu(rgb).mean()
    (frac_cracked if m.any() else frac_empty).append(f)

print(f"A1 predicted-positive fraction")
print(f"  on EMPTY   patches (n={len(frac_empty):4d}): {np.mean(frac_empty):.3f}")
print(f"  on CRACKED patches (n={len(frac_cracked):4d}): {np.mean(frac_cracked):.3f}")

**Diagnosis.** On empty patches A1 labels **44.1%** of all pixels as crack, versus
16.0% on patches that do contain one. Since 83.4% of patches are empty, these false
positives swamp the pooled precision.

The fix is therefore not a better morphological kernel - it is a threshold that is
**allowed to answer "nothing here"**. A per-patch adaptive threshold structurally
cannot; a single global threshold, learned once on the training split, can.

### Tuning the global threshold on train

We sweep the black-hat kernel size and threshold on a seeded sample of **training**
patches, then apply the winner to validation once.

In [ ]:
rng = np.random.default_rng(SEED)
train_patches = list(stream_patches(patches, images, "train"))
idx = rng.choice(len(train_patches), size=min(1200, len(train_patches)), replace=False)
tune = [(_clahe.apply(cv2.cvtColor(train_patches[i][0], cv2.COLOR_RGB2GRAY)),
         train_patches[i][1]) for i in idx]
del train_patches

grid = []
for kern in [21, 51, 101, 151]:
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kern, kern))
    resp = [(cv2.morphologyEx(g, cv2.MORPH_BLACKHAT, k), m) for g, m in tune]
    for thr in [80, 120, 160, 200]:
        TP = FP = FN = 0
        for bh, m in resp:
            p = bh > thr
            TP += int(np.count_nonzero(p & m))
            FP += int(np.count_nonzero(p & ~m))
            FN += int(np.count_nonzero(~p & m))
        grid.append(dict(kernel=kern, threshold=thr,
                         train_dice=2 * TP / max(2 * TP + FP + FN, 1)))

grid = pd.DataFrame(grid)
print(grid.pivot(index="kernel", columns="threshold", values="train_dice")
      .round(4).to_string())
best = grid.loc[grid.train_dice.idxmax()]
print(f"\nbest on train: kernel={int(best.kernel)} threshold={int(best.threshold)} "
      f"dice={best.train_dice:.4f}")

**Interpretation.** Performance rises steeply with kernel size and then flattens - the
gain from kernel 81 to 151 is under one percent relative. That plateau is informative:
once the structuring element is much wider than any crack, black-hat is effectively
performing **background subtraction**, and making it wider changes little.

We operate at **kernel 101, threshold 160**. That is within a fraction of a percent of
the best grid point while keeping the structuring element below 40% of the patch
width; at kernel 151 it spans 59% of a 256-pixel patch, which makes the operator
poorly defined near patch borders.

In [ ]:
BH_KERNEL, BH_THRESHOLD = 101, 160


def predict_blackhat_global(rgb, kernel=BH_KERNEL, thr=BH_THRESHOLD):
    """A3: black-hat with a global threshold learned on the training split."""
    return clean_components(blackhat_response(rgb, kernel) > thr)


rows = []
for name, fn in [
        ("A0 predict all background", predict_nothing),
        ("A1 Otsu on inverted gray", predict_otsu),
        ("A2 blackhat + per-patch Otsu", predict_blackhat_otsu),
        (f"A3 blackhat k={BH_KERNEL} + global thr={BH_THRESHOLD}",
         predict_blackhat_global)]:
    t0 = time.time()
    res = evaluate((fn(rgb), m) for rgb, m in stream_patches(patches, images, "val"))
    res.update(method=name, seconds=round(time.time() - t0, 1))
    rows.append(res)

classical = pd.DataFrame(rows).set_index("method")
show_cols = ["micro_dice", "micro_iou", "micro_precision", "micro_recall",
             "cracked_dice", "macro_dice", "pixel_accuracy"]
print(f"validation: {int(classical.n_patches.iloc[0]):,} patches, "
      f"{int(classical.n_cracked.iloc[0])} contain crack\n")
print(classical[show_cols].round(4).to_string())

**Results, and the two lessons.**

**Lesson one - pixel accuracy is meaningless here.** Look at the A0 row: predicting
*nothing at all* scores **98.6% pixel accuracy** and **0.838 macro Dice**, while micro
Dice is exactly 0. Any report that led with accuracy, or with unrestricted macro Dice,
would describe this useless model as excellent. This is the empirical justification
for the metric choices in Chapter 11.

**Lesson two - the threshold mattered more than the morphology.** A2 (black-hat with
per-patch Otsu) actually scores *worse* than plain Otsu on micro Dice, despite using a
far better-suited operator, because it inherits Otsu's inability to abstain. Replacing
only the thresholding step with a globally tuned cut lifts micro Dice from 0.046 to
**0.412** - a nine-fold improvement from the same morphological response. The
representation was never the bottleneck; the decision rule was.

A3 is a genuinely competent classical baseline, not a straw man, which is what makes
the comparison in 12.3 meaningful.

### 12.2 Baseline B - per-pixel classification on a filter bank

In [ ]:
SIGMAS = [1, 2, 4, 8]


def filter_bank(rgb):
    """Named multiscale per-pixel features. Returns (H, W, F) and the F names."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    feats, names = [gray], ["gray"]

    mu = cv2.blur(gray, (5, 5))
    sq = cv2.blur(gray * gray, (5, 5))
    feats.append(np.sqrt(np.maximum(sq - mu * mu, 0))); names.append("local_std_5")

    for s in SIGMAS:
        sm = cv2.GaussianBlur(gray, (0, 0), s)
        feats.append(sm); names.append(f"gauss_s{s}")
        gx = cv2.Sobel(sm, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(sm, cv2.CV_32F, 0, 1, ksize=3)
        feats.append(np.sqrt(gx * gx + gy * gy)); names.append(f"grad_mag_s{s}")
        feats.append(cv2.Laplacian(sm, cv2.CV_32F)); names.append(f"log_s{s}")
        hxx, hxy, hyy = hessian_matrix(gray, sigma=s, order="rc",
                                       use_gaussian_derivatives=False)
        big, small = hessian_matrix_eigvals([hxx, hxy, hyy])
        feats.append(big.astype(np.float32)); names.append(f"hess_max_s{s}")
        feats.append(small.astype(np.float32)); names.append(f"hess_min_s{s}")

    feats.append(cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)[:, :, 1].astype(np.float32))
    names.append("saturation")
    return np.stack(feats, axis=-1), names

In [ ]:
PIXELS_PER_PATCH, N_TRAIN_PATCHES = 800, 400

rng = np.random.default_rng(SEED)
sel = patches[patches.split == "train"]
cracked_sel = sel[sel.has_crack]
empty_sel = sel[~sel.has_crack]
take_c = min(len(cracked_sel), N_TRAIN_PATCHES // 2)
chosen = pd.concat([cracked_sel.sample(take_c, random_state=SEED),
                    empty_sel.sample(min(len(empty_sel), N_TRAIN_PATCHES - take_c),
                                     random_state=SEED)])

p_lookup = images.set_index("image_id")[["rgb_path", "mask_path"]]
Xs, ys, gs = [], [], []
t0 = time.time()
for image_id, grp in chosen.groupby("image_id"):
    row = p_lookup.loc[image_id]
    rgb_full, mask_full = load_rgb(row.rgb_path), load_mask(row.mask_path)
    for r in grp.itertuples():
        F, FEAT_NAMES = filter_bank(rgb_full[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size])
        lab = mask_full[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size].ravel()
        F = F.reshape(-1, F.shape[-1])
        pos, neg = np.flatnonzero(lab), np.flatnonzero(~lab)
        k = PIXELS_PER_PATCH // 2
        tp_ = rng.choice(pos, min(k, len(pos)), replace=False) if len(pos) else np.array([], int)
        tn_ = rng.choice(neg, min(PIXELS_PER_PATCH - len(tp_), len(neg)), replace=False)
        sel_idx = np.concatenate([tp_, tn_])
        Xs.append(F[sel_idx]); ys.append(lab[sel_idx])
        gs.append(np.full(len(sel_idx), int(image_id)))

Xpix, ypix, gpix = np.concatenate(Xs), np.concatenate(ys), np.concatenate(gs)
print(f"{Xpix.shape[0]:,} pixels x {Xpix.shape[1]} features from {len(chosen)} patches "
      f"({100*ypix.mean():.1f}% crack) in {time.time()-t0:.1f}s")
print(f"features: {FEAT_NAMES}")

In [ ]:
pixel_models = {
    "B1 LogisticRegression": make_pipeline(
        StandardScaler(), LogisticRegression(max_iter=2000, random_state=SEED)),
    "B2 RandomForest": RandomForestClassifier(
        n_estimators=60, min_samples_leaf=8, n_jobs=-1, random_state=SEED),
}

cv_rows = []
for name, mdl in pixel_models.items():
    folds = []
    for tr, te in GroupKFold(n_splits=3).split(Xpix, ypix, gpix):
        m = clone(mdl).fit(Xpix[tr], ypix[tr])
        pr = m.predict(Xpix[te])
        tp = int(np.count_nonzero(pr & ypix[te]))
        fp = int(np.count_nonzero(pr & ~ypix[te]))
        fn = int(np.count_nonzero(~pr & ypix[te]))
        folds.append(2 * tp / max(2 * tp + fp + fn, 1))
    cv_rows.append(dict(model=name, fold_dice=[round(f, 4) for f in folds],
                        mean=round(float(np.mean(folds)), 4),
                        std=round(float(np.std(folds)), 4)))
print(pd.DataFrame(cv_rows).to_string(index=False))

**A warning about this number.** The cross-validated Dice of roughly 0.88 is computed
on the **balanced pixel sample** (24% crack by construction), not on real patches. It
is *not* a segmentation score and must never be compared with the 0.412 above.

It is still worth having: because the folds are grouped by source image, it tells us
the per-pixel decision rule **generalises across images** (fold-to-fold spread is
small), which is a different and useful question from how the model performs on the
true, wildly imbalanced pixel distribution. That question is answered next.

In [ ]:
def predict_pixel_model(model, rgb):
    F, _ = filter_bank(rgb)
    h, w, f = F.shape
    return model.predict_proba(F.reshape(-1, f))[:, 1].reshape(h, w)


pixel_rows = []
for name, mdl in pixel_models.items():
    t0 = time.time()
    mdl.fit(Xpix, ypix)
    res = evaluate((predict_pixel_model(mdl, rgb) > 0.5, m)
                   for rgb, m in stream_patches(patches, images, "val"))
    res.update(method=name, seconds=round(time.time() - t0, 1))
    pixel_rows.append(res)
    print(f"{name}: micro_dice {res['micro_dice']:.4f}  "
          f"iou {res['micro_iou']:.4f}  ({res['seconds']}s)")

pixel_ml = pd.DataFrame(pixel_rows).set_index("method")

**Result.** On the real validation grid, the per-pixel logistic regression reaches
**micro Dice 0.572** - clearly better than the tuned classical baseline at 0.412. The
random forest is slightly worse and several times slower at inference, since it must
evaluate 60 trees for each of 97 million pixels.

That logistic regression beats random forest is worth a comment: with 23 well-chosen,
physically meaningful features, the decision boundary is close to linear, and the
forest's extra flexibility mostly buys variance. It is a reminder that "more powerful
model" and "better model" are not the same claim.

### 12.3 The U-Net

A **smoke-train**, not a final model: the goal is to prove the pipeline runs end to
end, measure the real cost per epoch so the final project can be planned, and get a
first honest reading. Training uses balanced patches; validation uses the complete
unsampled grid so the number is comparable to the baselines above.

In [ ]:
BATCH, EPOCHS, LR, BASE_WIDTH = 16, 15, 1e-3, 16


class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False),
            nn.BatchNorm2d(cout), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """Four-level U-Net.

    Pooling gives the network the context it needs to tell a crack from a scratch,
    but it also destroys the spatial precision needed to outline a two-pixel-wide
    structure. The skip connections resolve that tension by handing each decoder
    level the un-pooled features from the matching encoder level.
    """

    def __init__(self, base=BASE_WIDTH):
        super().__init__()
        w = [base, base * 2, base * 4, base * 8]
        self.d1, self.d2, self.d3 = (DoubleConv(3, w[0]), DoubleConv(w[0], w[1]),
                                     DoubleConv(w[1], w[2]))
        self.bottleneck = DoubleConv(w[2], w[3])
        self.u3, self.c3 = nn.ConvTranspose2d(w[3], w[2], 2, 2), DoubleConv(w[3], w[2])
        self.u2, self.c2 = nn.ConvTranspose2d(w[2], w[1], 2, 2), DoubleConv(w[2], w[1])
        self.u1, self.c1 = nn.ConvTranspose2d(w[1], w[0], 2, 2), DoubleConv(w[1], w[0])
        self.head, self.pool = nn.Conv2d(w[0], 1, 1), nn.MaxPool2d(2)

    def forward(self, x):
        s1 = self.d1(x)
        s2 = self.d2(self.pool(s1))
        s3 = self.d3(self.pool(s2))
        b = self.bottleneck(self.pool(s3))
        x = self.c3(torch.cat([self.u3(b), s3], 1))
        x = self.c2(torch.cat([self.u2(x), s2], 1))
        x = self.c1(torch.cat([self.u1(x), s1], 1))
        return self.head(x)


def dice_loss(logits, target, eps=1.0):
    p = torch.sigmoid(logits)
    num = 2 * (p * target).sum((1, 2, 3)) + eps
    den = p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + eps
    return 1 - (num / den).mean()


def combined_loss(logits, target):
    """BCE keeps per-pixel probabilities calibrated; Dice is insensitive to class
    frequency and so keeps gradient flowing on the 1.1% of pixels that are crack."""
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)

In [ ]:
def cache_split(split, balanced):
    """Load patches into RAM. Also returns the metadata rows in the SAME order as
    the arrays, so any prediction can be traced back to its patch and its cluster."""
    rng = np.random.default_rng(SEED)
    sel = patches[patches.split == split]
    if balanced:
        pos, neg = sel[sel.has_crack], sel[~sel.has_crack]
        sel = pd.concat([pos, neg.sample(min(len(neg), len(pos)), random_state=SEED)])
    imgs, msks, meta = [], [], []
    for image_id, grp in sel.groupby("image_id"):
        row = p_lookup.loc[image_id]
        rgb, mask = load_rgb(row.rgb_path), load_mask(row.mask_path)
        for r in grp.itertuples():
            imgs.append(rgb[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size])
            msks.append(mask[r.y0:r.y0 + r.size, r.x0:r.x0 + r.size])
            meta.append(r._asdict())
    order = rng.permutation(len(imgs))
    meta_df = pd.DataFrame(meta).iloc[order].reset_index(drop=True)
    return np.asarray(imgs, np.uint8)[order], np.asarray(msks, bool)[order], meta_df


tr_x, tr_y, _ = cache_split("train", balanced=True)
va_x, va_y, va_meta = cache_split("val", balanced=False)
print(f"train {tr_x.shape}  {100*tr_y.any((1,2)).mean():.1f}% contain crack (balanced)")
print(f"val   {va_x.shape}  {100*va_y.any((1,2)).mean():.1f}% contain crack "
      f"(complete unsampled grid)")
print(f"{(tr_x.nbytes + va_x.nbytes)/1e9:.2f} GB cached in RAM")

In [ ]:
def batches(imgs, msks, batch, shuffle, augment, rng):
    order = rng.permutation(len(imgs)) if shuffle else np.arange(len(imgs))
    for i in range(0, len(order), batch):
        sl = order[i:i + batch]
        x, y = imgs[sl].astype(np.float32) / 255.0, msks[sl].astype(np.float32)
        if augment:
            if rng.random() < 0.5: x, y = x[:, :, ::-1], y[:, :, ::-1]
            if rng.random() < 0.5: x, y = x[:, ::-1], y[:, ::-1]
            k = int(rng.integers(4))
            if k: x, y = np.rot90(x, k, (1, 2)), np.rot90(y, k, (1, 2))
        xt = torch.from_numpy(np.ascontiguousarray(x.transpose(0, 3, 1, 2)))
        yt = torch.from_numpy(np.ascontiguousarray(y))[:, None]
        yield xt.to(DEVICE), yt.to(DEVICE)


@torch.no_grad()
def predict_probs(model, imgs, batch=32):
    model.eval()
    out = np.empty((len(imgs), PATCH, PATCH), np.float32)
    for i in range(0, len(imgs), batch):
        x = imgs[i:i + batch].astype(np.float32) / 255.0
        xt = torch.from_numpy(x.transpose(0, 3, 1, 2)).to(DEVICE)
        with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
            p = torch.sigmoid(model(xt))
        out[i:i + batch] = p[:, 0].float().cpu().numpy()
    return out


def micro_scores(probs, masks, thr):
    pred = probs > thr
    tp = int(np.count_nonzero(pred & masks))
    fp = int(np.count_nonzero(pred & ~masks))
    fn = int(np.count_nonzero(~pred & masks))
    return scores_from_counts(tp, fp, fn)

In [ ]:
torch.manual_seed(SEED)
model = UNet().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))
rng = np.random.default_rng(SEED)

print(f"U-Net base width {BASE_WIDTH}, "
      f"{sum(p.numel() for p in model.parameters()):,} parameters\n")

best_dice, best_epoch, best_state, history = -1.0, -1, None, []
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); losses = []
    for x, y in batches(tr_x, tr_y, BATCH, True, True, rng):
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
            loss = combined_loss(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        losses.append(loss.item())
    sched.step()
    secs = time.time() - t0

    s = micro_scores(predict_probs(model, va_x), va_y, 0.5)
    if s["dice"] > best_dice:
        best_dice, best_epoch = s["dice"], epoch
        best_state = copy.deepcopy(model.state_dict())
    history.append(dict(epoch=epoch, train_loss=float(np.mean(losses)),
                        val_dice=s["dice"], val_iou=s["iou"], seconds=secs))
    print(f"epoch {epoch:2d}  loss {np.mean(losses):.4f}  val_dice {s['dice']:.4f}  "
          f"val_iou {s['iou']:.4f}  {secs:.1f}s"
          + ("   <- best" if epoch == best_epoch else ""))

history = pd.DataFrame(history)
model.load_state_dict(best_state)
print(f"\nbest epoch {best_epoch}, val Dice {best_dice:.4f} | "
      f"mean {history.seconds.mean():.1f}s/epoch")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(history.epoch, history.train_loss, "o-", color="#4C72B0")
axes[0].set_title("Training loss (BCE + Dice) decreases smoothly")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss")

axes[1].plot(history.epoch, history.val_dice, "o-", label="Dice", color="#55A868")
axes[1].plot(history.epoch, history.val_iou, "o-", label="IoU", color="#C44E52")
axes[1].axvline(best_epoch, ls="--", color="grey", label=f"best epoch {best_epoch}")
axes[1].set_title("Validation performance"); axes[1].set_xlabel("epoch")
axes[1].legend()
plt.show()

**Result.** The model converges cleanly and reaches a validation micro Dice around
**0.87**, far above every baseline. Training costs about **15 seconds per epoch** on
an RTX 3060 laptop GPU, so the full 458-image training run planned for the final
project is comfortably affordable - which is exactly the feasibility question this
smoke-train existed to answer.

**One finding worth recording.** An earlier run of this same configuration *without*
the cosine learning-rate schedule was unstable: validation Dice climbed to 0.85 by
epoch 8, then collapsed to 0.22 by epoch 13 while the training loss kept falling -
overfitting a 2,216-patch training set drawn from just 42 images. Adding cosine
annealing plus best-checkpoint selection removed the collapse. We record this because
it is a real result about how fragile training is at this data scale, and it directly
motivates using all 458 images in the final project.

In [ ]:
probs_val = predict_probs(model, va_x)
sweep = pd.DataFrame([dict(threshold=round(float(t), 2),
                           **micro_scores(probs_val, va_y, t))
                      for t in np.arange(0.10, 0.95, 0.05)])
best_row = sweep.loc[sweep.dice.idxmax()]

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(sweep.threshold, sweep.dice, "o-", label="Dice")
ax.plot(sweep.threshold, sweep.precision, "s--", label="precision", alpha=0.7)
ax.plot(sweep.threshold, sweep.recall, "^--", label="recall", alpha=0.7)
ax.axvline(best_row.threshold, color="crimson", ls=":",
           label=f"chosen {best_row.threshold:.2f}")
ax.set_xlabel("probability threshold"); ax.set_title(
    "Threshold selected on validation (test never used)")
ax.legend(); plt.show()

print(sweep.round(4).to_string(index=False))
UNET_THRESHOLD = float(best_row.threshold)

**Interpretation.** Dice is fairly flat near the optimum, meaning the model is not
critically sensitive to threshold choice - a good sign for robustness. Precision and
recall cross as expected: a low threshold catches more crack pixels at the cost of
false positives.

**Methodological caveat, stated plainly.** We selected both the *epoch* and the
*threshold* using validation, and then reported validation Dice. That makes the ~0.87
figure **optimistically biased** - it is a best case, not an unbiased estimate. The
correct number will come from the untouched test split in the final report, and we
expect it to be somewhat lower.

### 12.4 Comparing all models

In [ ]:
unet_full = evaluate((probs_val[i] > UNET_THRESHOLD, va_y[i])
                     for i in range(len(va_y)))
unet_full["method"] = f"C U-Net (thr {UNET_THRESHOLD:.2f}, epoch {best_epoch})"

comparison = pd.concat([classical, pixel_ml,
                        pd.DataFrame([unet_full]).set_index("method")])
final_cols = ["micro_dice", "micro_iou", "micro_precision", "micro_recall",
              "cracked_dice", "pixel_accuracy"]
print(comparison[final_cols].round(4).to_string())

fig, ax = plt.subplots(figsize=(9, 3.6))
comparison[["micro_dice", "micro_iou"]].plot.barh(ax=ax, width=0.75)
ax.set_xlabel("score on validation"); ax.set_title(
    "Segmentation quality: classical -> per-pixel ML -> U-Net")
ax.invert_yaxis(); plt.show()

**Interpretation.** The progression is clear and monotone in the right direction:

- Classical morphology with a properly tuned global threshold: **0.41** Dice.
- A learned per-pixel decision on hand-designed multiscale features: **0.57**.
- A U-Net learning both features and decision, with spatial context: **0.87**.

Each step adds one capability, and each buys a real improvement. The gap between B and
C is the most informative: both see the same pixels, but only the U-Net sees a
*neighbourhood large enough to establish that a dark line is a crack and not a shadow
edge or a stain boundary*. Context, not raw feature quality, is what the deep model
contributes.

Note also that the pixel-accuracy column is essentially constant across every model
including the do-nothing one, which is a final reminder of why it is excluded.

### 12.5 Qualitative results and failure cases

Numbers hide the shape of an error. These are validation patches with the highest and
lowest per-patch Dice, shown in the four-panel format we will use throughout.

In [ ]:
per_patch_dice = np.array([
    scores_from_counts(*confusion(probs_val[i] > UNET_THRESHOLD, va_y[i])[:3])["dice"]
    for i in range(len(va_y))])
cracked_idx = np.flatnonzero(va_y.any((1, 2)))
ranked = cracked_idx[np.argsort(per_patch_dice[cracked_idx])]


def show_examples(indices, title):
    fig, axes = plt.subplots(len(indices), 4, figsize=(11, 2.7 * len(indices)))
    for r, i in enumerate(indices):
        pred = probs_val[i] > UNET_THRESHOLD
        overlay = va_x[i].copy()
        overlay[va_y[i]] = (0.45 * overlay[va_y[i]] +
                            0.55 * np.array([0, 255, 0])).astype(np.uint8)
        overlay[pred] = (0.45 * overlay[pred] +
                         0.55 * np.array([255, 0, 0])).astype(np.uint8)
        for c, (img, cmap, lab) in enumerate([
                (va_x[i], None, "image"), (va_y[i], "gray", "ground truth"),
                (pred, "gray", "prediction"),
                (overlay, None, "overlay: green=truth red=pred")]):
            axes[r, c].imshow(img, cmap=cmap); axes[r, c].axis("off")
            if r == 0:
                axes[r, c].set_title(lab, fontsize=10)
        axes[r, 0].text(-0.08, 0.5, f"Dice {per_patch_dice[i]:.2f}", rotation=90,
                        va="center", ha="center", transform=axes[r, 0].transAxes,
                        fontsize=9)
    fig.suptitle(title, fontsize=12); plt.show()


show_examples(ranked[-4:][::-1], "Best predictions on crack-bearing validation patches")

In [ ]:
show_examples(ranked[:4], "Worst predictions - the failure modes")

**Interpretation of the failures.** The worst cases are not random; they fall into
recognisable groups, and each points at a different fix:

- **Very thin, low-contrast cracks** - hairline cracks only a pixel or two wide, where
  even a human needs to zoom. The model finds part of the crack and drops the faint
  continuation, so recall collapses while precision stays reasonable.
- **Cracks at a patch border**, cut by our grid so only a fragment is visible. This is
  an artefact of our own patching, and it is the strongest argument for the
  **overlapping tiled inference** planned in Chapter 16.
- **Dark linear distractors** - mortar joints, form-work lines, shadows in surface
  relief and staining. These are genuinely crack-like at the local level, and they are
  where a larger receptive field should help most.
- **Annotation disagreement** - cases where the prediction traces a visible crack that
  the mask does not mark, or where the mask is noticeably thicker than the crack. Here
  the "error" is partly in the ground truth, which caps the achievable Dice.

### 12.6 Performance by visual regime

A single average hides whether the model works everywhere or only on easy patches. We
now use the Chapter 9 clusters as evaluation strata.

In [ ]:
# The cluster model is fitted once on every patch, then applied to the validation
# patches in the exact order the prediction array uses (va_meta guarantees this).
cluster_scaler = StandardScaler().fit(patches[FEATURES].values)
cluster_model = KMeans(n_clusters=best_k, n_init=10, random_state=SEED).fit(
    cluster_scaler.transform(patches[FEATURES].values))
va_cluster = cluster_model.predict(cluster_scaler.transform(va_meta[FEATURES].values))
assert len(va_cluster) == len(va_y), "cluster labels must align with predictions"

strata_rows = []
for c in range(best_k):
    idx = np.flatnonzero(va_cluster == c)
    if len(idx) == 0:
        continue
    res = evaluate((probs_val[i] > UNET_THRESHOLD, va_y[i]) for i in idx)
    strata_rows.append(dict(
        cluster=c, patches=len(idx),
        with_crack=int(sum(va_y[i].any() for i in idx)),
        brightness=round(float(va_meta.gray_mean.values[idx].mean()), 1),
        texture=round(float(va_meta.glcm_contrast.values[idx].mean()), 3),
        micro_dice=round(res["micro_dice"], 4),
        precision=round(res["micro_precision"], 4),
        recall=round(res["micro_recall"], 4)))

strata = pd.DataFrame(strata_rows)
print(strata.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.bar(strata.cluster.astype(str), strata.micro_dice, color="#4C72B0")
ax.set_xlabel("visual regime (cluster from chapter 9)")
ax.set_ylabel("micro Dice")
ax.set_title("U-Net performance is not uniform across visual regimes")
plt.show()

**Interpretation.** The breakdown shows performance is **not uniform across visual
regimes**. Clusters differ both in how many crack pixels they contain and in how well
the model recovers them, and the `brightness` and `texture` columns identify which
kind of surface each cluster represents.

This is the actionable form of an error analysis. A single average Dice would report
one number and hide the fact that the model is markedly stronger on some surface types
than others; the stratified view tells the final project **where** to spend
augmentation and data-collection effort. Clusters with few crack-bearing patches
should be read cautiously, since their Dice rests on a small pixel count.

---
# 13. Explaining the models

An explanation should answer a question we actually have. Ours is: **which visual
evidence does a model use to decide a pixel is crack, and is that evidence physically
sensible?** If a model relies on brightness alone, it will fail on shadows; if it uses
multiscale ridge structure, it is doing something defensible.

In [ ]:
rf = pixel_models["B2 RandomForest"]
sub = rng.choice(len(Xpix), size=min(30000, len(Xpix)), replace=False)
perm = permutation_importance(rf, Xpix[sub], ypix[sub], n_repeats=5,
                              random_state=SEED, n_jobs=-1)

imp = pd.DataFrame({
    "impurity": rf.feature_importances_,
    "permutation": perm.importances_mean,
}, index=FEAT_NAMES).sort_values("permutation", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5.5))
imp.head(15)[::-1].plot.barh(ax=ax, width=0.8)
ax.set_title("Which filter, at which scale, detects cracks?")
ax.set_xlabel("importance")
plt.show()
print(imp.round(4).to_string())

**Interpretation.** The two importance measures broadly agree, which matters because
impurity importance is known to favour high-cardinality continuous features while
permutation importance measures actual predictive contribution.

The ranking is physically sensible: **smoothed intensity and gradient magnitude at
several scales** dominate, with **Hessian (ridge) responses** and local standard
deviation contributing. The presence of multiple *scales* among the top features is
the important part - it says the model is not using a single-scale edge rule but
combining evidence about structures of different widths, which is exactly what one
would design by hand for cracks of varying thickness. Saturation contributes little,
consistent with cracks being achromatic.

This also explains **why the U-Net wins**: the filter bank is fixed and local, capped
at sigma = 8, whereas the U-Net learns its filters and sees a much larger context
through pooling.

### Planned for the final report

- **SHAP** values on the pixel model, to move from "which feature matters on average"
  to "why was *this* pixel called crack".
- **Occlusion sensitivity** for the U-Net: slide a grey square over the input and
  measure the drop in predicted crack probability, producing a map of which image
  regions the decision depended on.
- **Encoder feature-map visualisation**, to check that early layers respond to
  oriented edges and deeper layers to elongated structures.
- The **per-regime error breakdown** from 12.6 as a global explanation.

---
# 14. Tools and environment

| Purpose | Tool |
|---|---|
| Language | Python 3.14 |
| Deep learning | PyTorch 2.13 with CUDA 12.6, AMP mixed precision |
| Classical ML | scikit-learn (LogisticRegression, RandomForest, KMeans, DBSCAN, Agglomerative, PCA, t-SNE, GroupKFold) |
| Image processing | OpenCV (CLAHE, morphology, Canny, connected components), scikit-image (Hessian, GLCM) |
| Data | pandas, NumPy, PyArrow (parquet) |
| Visualization | matplotlib, seaborn |
| Dimensionality reduction | scikit-learn PCA and t-SNE, umap-learn |
| Explainability | scikit-learn permutation importance; SHAP planned |
| Near-duplicate detection | ImageHash (perceptual hash) |
| Environment | conda env `kaggle`, JupyterLab |
| Hardware | NVIDIA RTX 3060 Laptop, 6 GB VRAM |

Exact pinned versions are in `requirements.txt`. Beyond the workshop material, this
project required independently learning semantic segmentation, U-Net architectures,
overlap-based losses, morphological image processing, and EXIF metadata handling.

---
# 15. Limitations

Stated honestly, because most of these bound how far the results can be trusted.

**Data**

1. **458 images from a single campus.** One geography, one construction era, one
   camera. Generalisation to other concrete, other lighting or other cameras is
   untested and probably optimistic.
2. **No crack-free source images.** Every one of the 458 photographs contains a crack.
   The model has never seen a photograph of an intact wall as a whole, only crack-free
   *crops*. Deployment on genuinely clean surfaces is therefore extrapolation.
3. **Lossy JPEG masks.** 0.52% of mask pixels are ambiguous mid-greys; our threshold
   of 127 assigns them, and a different threshold would change every Dice score here.
4. **Single-annotator ground truth.** No inter-annotator agreement is available, so we
   cannot separate model error from annotation error, and the masks are visibly
   thicker than the cracks. This places an unknown ceiling on achievable Dice.

**Method**

5. **The proposal uses a 60-image subset** for everything from patch generation
   onward. The split covers all 458, but every model number comes from 42 training and
   9 validation images.
6. **Validation-selected, validation-reported.** The U-Net epoch and threshold were
   chosen on validation and the score reported on it, so ~0.87 is optimistic.
7. **Patch-level, not image-level, evaluation.** Real use requires tiling a whole
   photograph and stitching; border effects at patch seams are not yet measured.
8. **No cross-validation of the U-Net.** Only the cheap pixel model is
   cross-validated; the U-Net rests on one split, so we have no variance estimate.
9. **A single architecture and patch size.** No ablation of 256 vs 512, of downscaling,
   of encoder depth, or of loss weighting yet.

---
# 16. Summary and plan for the final project

### What this proposal established

1. A working, verified data pipeline over all 458 image/mask pairs.
2. **A serious data defect found and fixed**: EXIF orientation misaligns 194 of 458
   images, and 15 of them cannot be caught by any dimension check. We built a
   content-based alignment test that confirms correct alignment on 458/458.
3. Quantified the target: crack pixels are ~1.1% of all pixels; masks are not truly
   binary; 83.4% of 256x256 patches are empty.
4. A leakage-safe, stratified, **frozen** 320/69/69 split at the level of
   near-duplicate groups, with the duplicate threshold chosen from evidence rather
   than folklore.
5. An EDA showing hand-crafted descriptors **do not** separate cracked from clean
   patches - the argument for learned features - plus clusters that serve as
   evaluation strata.
6. Three families of working segmentation model with real numbers:
   classical **0.41**, per-pixel ML **0.57**, U-Net **0.87** micro Dice on validation.
7. A measured compute budget: **15 s/epoch**, so full-scale training is affordable.

### Open questions raised by this analysis

- How much of the U-Net's advantage survives when the classical baseline gets
  overlapping tiled inference rather than independent patches?
- Is the annotation-thickness bias systematically depressing precision, and would
  evaluating on a dilated ground truth change the ranking?
- Does the model detect cracks, or dark linear structures generally? The failure cases
  suggest the latter is a real risk.

### Plan for the final project

**Data and scale**
- Raise `PROPOSAL_SUBSET_N` to all 458 images (~75,600 patches), reusing the frozen
  split.
- Overlapping training crops with random offsets for effective augmentation.

**Modelling**
- Train the U-Net to convergence with early stopping; add a pretrained ResNet encoder
  and compare against the from-scratch model.
- Ablate the loss (BCE, Dice, BCE+Dice, Focal+Dice) - the imbalance makes this the
  single most consequential hyperparameter.
- Ablate patch size (256 vs 512) and the downscale-then-crop alternative.

**Evaluation**
- Report on the **held-out test split**, once.
- **Tiled whole-image inference** with overlapping windows and averaged predictions,
  reporting both patch-level and whole-image Dice.
- k-fold cross-validation of the U-Net if compute allows, for a variance estimate.
- Full per-cluster stratified reporting and a systematic failure taxonomy.

**Explainability**
- SHAP on the pixel model; occlusion sensitivity and feature-map visualisation on the
  U-Net.

**Post-processing**
- Morphological cleanup and small-component removal on predictions, and measure
  whether it helps or merely trades recall for precision.

In [ ]:
history.to_csv(OUT / "unet_history.csv", index=False)
sweep.to_csv(OUT / "unet_threshold_sweep.csv", index=False)
comparison.to_csv(OUT / "model_comparison.csv")
torch.save({"state_dict": model.state_dict(), "threshold": UNET_THRESHOLD,
            "base_width": BASE_WIDTH, "best_epoch": int(best_epoch)},
           OUT / "unet_smoke.pt")

print("artifacts written to outputs/:")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:32s} {f.stat().st_size/1024:8.1f} KB")
print("\nProposal notebook complete.")